# AC SugarSims: Architecture Experiment
**18 runs** | 6 conditions x 3 seeds x 3,000 steps | Parallel execution

Run all cells in order.

In [ ]:
# ── 1. Install + Clone ──
!pip install -q mesa numpy pandas pyarrow

import os
if os.path.exists('ac_sugarsims'):
    %cd ac_sugarsims
    !git pull
else:
    !git clone https://github.com/caseymrobbins/ac_sugarsims.git
    %cd ac_sugarsims

print(f"Dir: {os.getcwd()}")
!ls *.py | head -5


In [ ]:
# ── 2. Verify modules ──
import importlib
for m in ['environment','agents','planner','metrics','trust',
          'innovation','sustainable_capitalism','horizon_index']:
    try:
        importlib.import_module(m)
        print(f"  OK {m}")
    except Exception as e:
        print(f"  FAIL {m}: {e}")


In [ ]:
# ── 3. Smoke test ──
from environment import EconomicModel
import time
model = EconomicModel(seed=42, grid_width=40, grid_height=40,
                      n_workers=50, n_firms=5, n_landowners=3, objective="SUM_RAW")
t0 = time.time()
for i in range(100): model.step()
elapsed = time.time() - t0
print(f"100 steps in {elapsed:.1f}s (est {elapsed/100*3000/60:.0f} min per 3000-step run)")
print("Smoke test passed")


In [ ]:
# ── 4. RUN EXPERIMENT ──
import multiprocessing, time
n_workers = max(1, multiprocessing.cpu_count() - 1)
print(f"CPUs: {multiprocessing.cpu_count()}, using {n_workers} workers")
t0 = time.time()
!python run_parallel.py --workers $n_workers
elapsed = time.time() - t0
print(f"\nWall time: {elapsed/60:.1f} min")


In [ ]:
# ── 5. Load results ──
import pandas as pd, os
raw_dir = "results/architecture/raw_data"
pfiles = sorted([f for f in os.listdir(raw_dir) if f.endswith('.parquet')])
print(f"{len(pfiles)} files")
all_data = pd.concat([pd.read_parquet(f"{raw_dir}/{f}") for f in pfiles], ignore_index=True)
print(f"Rows: {len(all_data)}, Conditions: {sorted(all_data['condition'].unique())}")


In [ ]:
# ── 6. Comparison ──
import numpy as np
late = all_data[all_data['step'] >= 2500]
conds = sorted(late['condition'].unique())
short = {'C1_vanilla_sum':'C1:Van','C2_sevc_sum':'C2:SEVC','C3_sevc_hi_sum':'C3:+HI',
         'C4_sevc_trust_sum':'C4:+Tr','C5_sevc_trust_hi_sum':'C5:T+HI','C6_full_topo':'C6:TOPO'}
metrics = [('worker_min','Floor','h'),('worker_mean','Mean','h'),('worker_gini','WGini','l'),
           ('unemployment_rate','Unemp','l'),('agency_floor','AgFloor','h'),('n_workers','Pop','h'),
           ('n_firms','Firms','h'),('n_active_cartels','Cartels','l'),('mean_skill','Skill','h'),
           ('total_production','Prod','h'),('horizon_index','HI','h'),('mean_firm_floor','FirmFl','h'),
           ('trust_institutional','Trust','h')]
hdr = f"{'Metric':<12}" + "".join(f" {short.get(c,c):>10}" for c in conds)
print(hdr); print("-"*len(hdr))
for k,lbl,d in metrics:
    if k not in late.columns: continue
    row = f"{lbl:<12}"
    vals = {c: late[late['condition']==c][k].mean() for c in conds}
    best = min(vals.values()) if d=='l' else max(vals.values())
    for c in conds:
        v = vals[c]; m = "*" if abs(v-best)<abs(best)*0.01+1e-9 else " "
        if abs(v)>=1000: row += f" {v:>9,.0f}{m}"
        elif abs(v)>=1: row += f" {v:>9.2f}{m}"
        else: row += f" {v:>9.4f}{m}"
    print(row)


In [ ]:
# ── 7. Staircase plot ──
import matplotlib.pyplot as plt
co = ['C1_vanilla_sum','C2_sevc_sum','C3_sevc_hi_sum','C4_sevc_trust_sum','C5_sevc_trust_hi_sum','C6_full_topo']
cl = ['C1:Vanilla','C2:+SEVC','C3:+HI','C4:+Trust','C5:+T+HI','C6:TOPO']
cc = ['#94a3b8','#6366f1','#8b5cf6','#f59e0b','#f97316','#10b981']
sm = {'worker_min':'Floor','unemployment_rate':'Unemp','agency_floor':'AgFloor','n_firms':'Firms',
      'n_active_cartels':'Cartels','mean_skill':'Skill','total_production':'Production','horizon_index':'HI'}
fig,axes = plt.subplots(2,4,figsize=(22,10))
fig.suptitle('Architecture Staircase',fontsize=16,fontweight='bold')
for idx,(k,t) in enumerate(sm.items()):
    if k not in late.columns: continue
    ax = axes[idx//4][idx%4]
    ms = [late[late['condition']==c].groupby('seed')[k].mean().mean() for c in co]
    ss = [late[late['condition']==c].groupby('seed')[k].mean().std() for c in co]
    ax.bar(range(6),ms,yerr=ss,color=cc,alpha=0.85,capsize=3)
    ax.set_xticks(range(6)); ax.set_xticklabels(cl,rotation=45,ha='right',fontsize=8)
    ax.set_title(t,fontsize=11,fontweight='bold'); ax.grid(axis='y',alpha=0.3)
plt.tight_layout(); plt.savefig('staircase.png',dpi=150,bbox_inches='tight'); plt.show()


In [ ]:
# ── 8. Download ──
import shutil
shutil.make_archive('architecture_results','zip','results/architecture')
from google.colab import files
files.download('architecture_results.zip')
